In [ ]:
!pip install "psycopg[binary]==3.3.6" "python-dotenv==1.2.4"


Installs the driver and the `.env` loader at pinned versions.


In [ ]:
# Bring in the tools we need.
import os
import threading
import time

import psycopg
from psycopg import sql
from dotenv import load_dotenv

load_dotenv("../../.env")
PGHOST = os.getenv("PGHOST", "localhost")
PGPORT = os.getenv("PGPORT", "5432")
PGUSER = os.getenv("PGUSER", "postgres")
PGPASSWORD = os.getenv("PGPASSWORD") or None
PGDATABASE = os.getenv("PGDATABASE", "university")
PGSSLMODE = os.getenv("PGSSLMODE", "prefer")
conn_args = dict(host=PGHOST, port=PGPORT, user=PGUSER,
                 password=PGPASSWORD, sslmode=PGSSLMODE)

try:
    with psycopg.connect(autocommit=True, **{**conn_args, "dbname": "postgres"}) as admin:
        if not admin.execute("SELECT 1 FROM pg_database WHERE datname = %s", (PGDATABASE,)).fetchone():
            admin.execute(sql.SQL("CREATE DATABASE {}").format(sql.Identifier(PGDATABASE)))
    conn = psycopg.connect(dbname=PGDATABASE, autocommit=True, **conn_args)
    print("connected to", PGDATABASE)
except psycopg.OperationalError as err:
    raise SystemExit(f"Could not connect. Is PostgreSQL running?\n{err}")


Open the main connection in autocommit mode so `CALL` to a procedure that does `COMMIT` behaves predictably, and so DDL we run here takes effect immediately.


In [ ]:
# Print query results as a tidy table.
def show(query, params=None, using=None):
    session = using or conn
    with session.cursor() as cur:
        cur.execute(query, params)
        if cur.description is None:
            print(cur.statusmessage)
            return
        cols = [d.name for d in cur.description]
        rows = cur.fetchall()
    cells = [["NULL" if v is None else str(v) for v in row] for row in rows]
    widths = [max([len(c)] + [len(row[i]) for row in cells]) for i, c in enumerate(cols)]
    print(" | ".join(c.center(w) for c, w in zip(cols, widths)).rstrip())
    print("-+-".join("-" * w for w in widths))
    for row in cells:
        print(" | ".join(v.ljust(w) for v, w in zip(row, widths)).rstrip())
    print(f"({len(rows)} row{'' if len(rows) == 1 else 's'})")


# A second connection for LISTEN/NOTIFY.
def new_session():
    return psycopg.connect(dbname=PGDATABASE, autocommit=True, **conn_args)


In [ ]:
# Start clean, then create tables for shows, bookings, audit history, and logs.
conn.execute("DROP EVENT TRIGGER IF EXISTS log_ddl")
conn.execute("DROP TABLE IF EXISTS bookings, events, audit_log, ddl_log, bookings_archive CASCADE")

conn.execute("""
    CREATE TABLE events (
        event_id  SERIAL PRIMARY KEY,
        name      TEXT NOT NULL,
        starts_at TIMESTAMPTZ NOT NULL
    )
""")
conn.execute("""
    CREATE TABLE bookings (
        booking_id  BIGSERIAL PRIMARY KEY,
        event_id    INTEGER NOT NULL REFERENCES events (event_id) ON DELETE CASCADE,
        customer    TEXT NOT NULL,
        seats       INTEGER NOT NULL,
        price_cents INTEGER NOT NULL,
        created_at  TIMESTAMPTZ NOT NULL DEFAULT now()
    )
""")
conn.execute("""
    CREATE TABLE audit_log (
        audit_id   BIGSERIAL PRIMARY KEY,
        table_name TEXT NOT NULL,
        action     TEXT NOT NULL,
        row_data   JSONB NOT NULL,
        changed_at TIMESTAMPTZ NOT NULL DEFAULT now()
    )
""")
conn.execute("CREATE TABLE bookings_archive (LIKE bookings INCLUDING ALL)")
conn.execute("""
    CREATE TABLE ddl_log (
        ddl_id  BIGSERIAL PRIMARY KEY,
        tag     TEXT NOT NULL,
        command TEXT NOT NULL,
        ran_at  TIMESTAMPTZ NOT NULL DEFAULT now()
    )
""")

conn.execute("""
    INSERT INTO events (name, starts_at) VALUES
        ('Jazz Night', now() + interval '7 days'),
        ('Stand-up', now() + interval '14 days')
""")
conn.execute("""
    INSERT INTO bookings (event_id, customer, seats, price_cents) VALUES
        (1, 'alice', 2, 9000),
        (1, 'bob', 1, 4500),
        (2, 'carol', 3, 12000)
""")

show("SELECT event_id, name FROM events ORDER BY event_id")
show("SELECT booking_id, customer, seats, price_cents FROM bookings ORDER BY booking_id")


- **Input:** nothing to load.
- **What happens:** create the tables, add seed data, and show it.
- **Output:** two events and three bookings.
- **Why:** a realistic schema to attach functions, triggers, and a procedure to.


In [ ]:
# A simple SQL function that gives a quantity discount (10% off for 3+ seats).
conn.execute("""
    CREATE OR REPLACE FUNCTION seat_price(base_cents int, seats int)
    RETURNS int
    LANGUAGE sql IMMUTABLE
    AS $$
        SELECT (base_cents * seats * CASE WHEN seats >= 3 THEN 90 ELSE 100 END) / 100;
    $$
""")

# A PL/pgSQL function that maps a price to a band.
conn.execute("""
    CREATE OR REPLACE FUNCTION price_band(cents int)
    RETURNS text
    LANGUAGE plpgsql IMMUTABLE
    AS $$
    BEGIN
        IF cents < 5000 THEN
            RETURN 'budget';
        ELSIF cents < 10000 THEN
            RETURN 'standard';
        ELSE
            RETURN 'premium';
        END IF;
    END;
    $$
""")

show("SELECT seat_price(4500, 2) AS two_seats, seat_price(4500, 4) AS four_seats")
show("SELECT booking_id, price_cents, price_band(price_cents) AS band FROM bookings ORDER BY booking_id")


- **Input:** integer cents and a seat count.
- **What happens:** `seat_price` is pure SQL; `price_band` uses PL/pgSQL's `IF/ELSIF/ELSE`. Both are marked `IMMUTABLE` because they depend only on their inputs.
- **Output:** `9000` and `16200`; bands `standard`, `budget`, `premium`.
- **Why:** functions let the database enforce the pricing rule once and use it anywhere.


In [ ]:
# Summarise bookings per event without writing a view first.
conn.execute("""
    CREATE OR REPLACE FUNCTION event_totals()
    RETURNS TABLE (event_name text, bookings bigint, seats bigint, revenue_cents bigint)
    LANGUAGE sql STABLE
    AS $$
        SELECT e.name, count(b.booking_id), COALESCE(sum(b.seats), 0), COALESCE(sum(b.price_cents), 0)
        FROM events e LEFT JOIN bookings b USING (event_id)
        GROUP BY e.name
        ORDER BY e.name;
    $$
""")

show("SELECT * FROM event_totals()")


- **Input:** none (reads from `events` and `bookings`).
- **What happens:** the function returns a table and you call it like any table in `SELECT`.
- **Output:** one row per event with counts and totals.
- **Why:** `RETURNS TABLE` is great for reusable reports; `STABLE` tells the planner the result is stable within a single statement.


In [ ]:
# A procedure that archives old bookings and commits inside the procedure.
conn.execute("""
    CREATE OR REPLACE PROCEDURE archive_old_bookings(older_than_days int)
    LANGUAGE plpgsql
    AS $$
    DECLARE
        moved int;
    BEGIN
        INSERT INTO bookings_archive
        SELECT * FROM bookings WHERE created_at < now() - make_interval(days => older_than_days);

        GET DIAGNOSTICS moved = ROW_COUNT;

        DELETE FROM bookings WHERE created_at < now() - make_interval(days => older_than_days);

        COMMIT;
        RAISE NOTICE 'archived % old bookings', moved;
    END;
    $$
""")

# Plant an old booking so there is something to move.
conn.execute("""
    INSERT INTO bookings (event_id, customer, seats, price_cents, created_at)
    VALUES (2, 'old-timer', 1, 3000, now() - interval '90 days')
""")

conn.execute("CALL archive_old_bookings(30)")
show("SELECT count(*) AS live_bookings FROM bookings")
show("SELECT booking_id, customer FROM bookings_archive ORDER BY booking_id")


- **Input:** a cutoff in days.
- **What happens:** `GET DIAGNOSTICS` captures how many rows were copied; `INSERT ... SELECT` moves them to the archive; `DELETE` removes them from live bookings; `COMMIT` ends that work; `RAISE NOTICE` prints a message.
- **Output:** live bookings drop by one, and the old booking appears in `bookings_archive`.
- **Why:** procedures can batch and commit their own work, which is safer for large cleanups than doing it all in one huge transaction.


In [ ]:
# A trigger function that records before/after as JSON and notifies listeners.
conn.execute("""
    CREATE OR REPLACE FUNCTION audit_booking_change()
    RETURNS trigger
    LANGUAGE plpgsql
    AS $$
    BEGIN
        INSERT INTO audit_log (table_name, action, row_data)
        VALUES (TG_TABLE_NAME, TG_OP,
                jsonb_build_object('old', to_jsonb(OLD), 'new', to_jsonb(NEW)));
        PERFORM pg_notify('booking_events',
                          TG_OP || ' ' || COALESCE(NEW.booking_id, OLD.booking_id)::text);
        RETURN COALESCE(NEW, OLD);
    END;
    $$
""")

conn.execute("DROP TRIGGER IF EXISTS bookings_audit ON bookings")
conn.execute("""
    CREATE TRIGGER bookings_audit
    AFTER INSERT OR UPDATE OR DELETE ON bookings
    FOR EACH ROW EXECUTE FUNCTION audit_booking_change()
""")

# Fire all three operations.
conn.execute("INSERT INTO bookings (event_id, customer, seats, price_cents) VALUES (1, 'dave', 2, 9000)")
conn.execute("UPDATE bookings SET seats = 4 WHERE customer = 'dave'")
conn.execute("DELETE FROM bookings WHERE customer = 'dave'")

show("""
    SELECT action,
           row_data->'new'->>'customer' AS new_customer,
           row_data->'old'->>'seats'    AS old_seats
    FROM audit_log ORDER BY audit_id
""")


- **Input:** changes to `bookings`.
- **What happens:** `TG_OP` tells us whether it was `INSERT`/`UPDATE`/`DELETE`; `OLD` and `NEW` give the row versions; `to_jsonb` stores them; `pg_notify` publishes an event. `AFTER` and returning `COALESCE(NEW, OLD)` is the standard pattern.
- **Output:** three audit rows (INSERT, UPDATE, DELETE). Old values appear on UPDATE/DELETE and new values on INSERT/UPDATE.
- **Why:** an audit trail answers "who changed what and when" without relying on every app to remember to log it.


In [ ]:
# Reject bookings with zero or negative seats before they are written.
conn.execute("""
    CREATE OR REPLACE FUNCTION check_booking()
    RETURNS trigger
    LANGUAGE plpgsql
    AS $$
    BEGIN
        IF NEW.seats <= 0 THEN
            RAISE EXCEPTION 'seats must be positive, got %', NEW.seats;
        END IF;
        RETURN NEW;
    END;
    $$
""")

conn.execute("DROP TRIGGER IF EXISTS bookings_check ON bookings")
conn.execute("""
    CREATE TRIGGER bookings_check
    BEFORE INSERT OR UPDATE ON bookings
    FOR EACH ROW EXECUTE FUNCTION check_booking()
""")

try:
    conn.execute("INSERT INTO bookings (event_id, customer, seats, price_cents) VALUES (1, 'erin', 0, 1000)")
except psycopg.errors.RaiseException as err:
    print("refused:", str(err).strip().splitlines()[0])

show("SELECT count(*) AS erin_bookings FROM bookings WHERE customer = 'erin'")


- **Input:** an invalid row with `seats = 0`.
- **What happens:** the `BEFORE` trigger checks the rule and calls `RAISE EXCEPTION`, which aborts the statement. The row is never inserted.
- **Output:** an exception message and zero bookings for `erin`.
- **Why:** database rules are the last line of defence — even if an app forgets to validate, PostgreSQL enforces them.


In [ ]:
# Record every CREATE/ALTER/DROP TABLE in ddl_log.
conn.execute("""
    CREATE OR REPLACE FUNCTION log_ddl_command()
    RETURNS event_trigger
    LANGUAGE plpgsql
    AS $$
    BEGIN
        INSERT INTO ddl_log (tag, command) VALUES (TG_TAG, current_query());
    END;
    $$
""")

conn.execute("DROP EVENT TRIGGER IF EXISTS log_ddl")
conn.execute("""
    CREATE EVENT TRIGGER log_ddl
    ON ddl_command_end
    WHEN TAG IN ('CREATE TABLE', 'ALTER TABLE', 'DROP TABLE')
    EXECUTE FUNCTION log_ddl_command()
""")

# Make a few schema changes to watch them be logged.
conn.execute("CREATE TABLE demo_table (id int)")
conn.execute("ALTER TABLE demo_table ADD COLUMN note text")
conn.execute("DROP TABLE demo_table")

show("SELECT tag, left(command, 38) AS command_start FROM ddl_log ORDER BY ddl_id")


- **Input:** three DDL statements.
- **What happens:** `TG_TAG` tells us which DDL command ran, `current_query()` captures the statement, and we insert into `ddl_log`. Only `CREATE TABLE`, `ALTER TABLE`, and `DROP TABLE` fire this trigger.
- **Output:** three rows in order of execution.
- **Why:** event triggers give visibility into who is changing the schema, without auditing every single table.


In [ ]:
# Open a listener on a second connection, then fire a change on the first.
listener = new_session()
listener.execute("LISTEN booking_events")

conn.execute("INSERT INTO bookings (event_id, customer, seats, price_cents) VALUES (2, 'frank', 2, 8000)")

for note in listener.notifies(timeout=3, stop_after=1):
    print("notification on", note.channel, ":", note.payload)

listener.close()


- **Input:** a new booking inserted after the listener subscribes.
- **What happens:** the audit trigger calls `pg_notify('booking_events', 'INSERT ...')`. Since we're in autocommit mode, the notify is delivered at commit; the listener picks it up and prints the payload.
- **Output:** `notification on booking_events : INSERT ...`
- **Why:** `LISTEN`/`NOTIFY` is a simple, zero-infrastructure way for processes to react to database changes.

```mermaid
sequenceDiagram
    participant A as App (writer)
    participant DB as PostgreSQL
    participant L as Listener
    L->>DB: LISTEN booking_events
    A->>DB: INSERT into bookings
    DB->>DB: Trigger calls pg_notify
    DB-->>L: notification with payload
```


In [ ]:
# Check what's available and install pgcrypto if missing.
show("""
    SELECT name, default_version, installed_version
    FROM pg_available_extensions
    WHERE name IN ('pgcrypto', 'pg_cron', 'postgis')
    ORDER BY name
""")

conn.execute("CREATE EXTENSION IF NOT EXISTS pgcrypto")

# Generate a UUID and hash a password.
show("SELECT gen_random_uuid() AS token")
show("SELECT crypt('correct horse', gen_salt('bf')) AS stored_hash")
show("""
    SELECT crypt('correct horse', stored_hash) = stored_hash AS password_matches
    FROM (SELECT crypt('correct horse', gen_salt('bf')) AS stored_hash) AS h
""")
show("SELECT encode(digest('postgres', 'sha256'), 'hex') AS sha256_of_postgres")


- **Input:** a password string.
- **What happens:** `CREATE EXTENSION pgcrypto` adds cryptographic functions. `gen_random_uuid()` makes a random UUID. `gen_salt('bf')` builds a blowfish salt; `crypt` hashes with it; comparing `crypt(plain, stored_hash)` to the stored hash tells us if it matches. `digest` computes a hash; `encode(..., 'hex')` prints it as hex.
- **Output:** a UUID, a hash starting with `$2a$`, `true`, and a 64-hex digest.
- **Why:** never roll your own password hashing — use a tested extension like `pgcrypto`. Always hash with a salt (here `gen_salt`) rather than storing plain text or unsalted digests.


In [ ]:
# See what we just created alongside extension functions.
show("""
    SELECT p.proname, l.lanname, p.prokind, p.provolatile,
           pg_get_function_identity_arguments(p.oid) AS arguments
    FROM pg_proc p
    JOIN pg_language l ON l.oid = p.prolang
    WHERE p.pronamespace = 'public'::regnamespace AND p.prokind IN ('f', 'p')
    ORDER BY p.proname
""")


- **Input:** the system catalogs `pg_proc` and `pg_language`.
- **What happens:** we list our functions (`prokind = 'f'`) and procedures (`prokind = 'p'`), their language, volatility, and arguments.
- **Output:** includes `seat_price`, `price_band`, `event_totals`, `archive_old_bookings`, and our trigger functions.
- **Why:** `pg_catalog` is the source of truth for what's installed; it's how you audit what logic lives in the database.


In [ ]:
# Drop the event trigger we created and tidy up.
conn.execute("DROP EVENT TRIGGER IF EXISTS log_ddl")
conn.execute("DROP TABLE IF EXISTS ddl_log")
conn.close()
print("connection closed — tables stay so you can explore further")
